In [28]:
import datetime
import numpy as np
import pandas as pd
from pandas_datareader import data as web
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

In [87]:
def sharpe_optimizer(symbols, holdings):
    # start and end date
    start_date = '2012-01-01'
    end_date = datetime.datetime.today().strftime('%Y-%m-%d')

    # get historical data on adjusted close for stocks in portfolio
    df = pd.DataFrame()
    for symbol in symbols:
        df[symbol] = web.DataReader(symbol, data_source='yahoo', start=start_date, end=end_date)['Adj Close']

    # calculate current return, risk and sharpe ratio
    returns = df.pct_change()
    current_weights = current_holdings/current_holdings.sum()
    current_return = np.sum(returns.mean() * current_weights) * 252
    pct_current_return = str(round(current_return* 100,2) ) + '%'
    cov_matrix_annual = returns.cov() * 252
    current_variance = np.dot(current_weights.T, np.dot(cov_matrix_annual, current_weights))
    current_risk = np.sqrt(current_variance)
    pct_current_risk = str(round(current_risk * 100,2)) + '%'
    current_sharpe = round((current_return - 0.02) / current_risk, 2)

    print('Current Portfolio Weights: ' )
    for symbol,weight in zip(symbols, current_weights):
        print(' * ' + symbol,weight)
    print('Current Portfolio Return: ' + str(pct_current_return))
    print('Current Portfolio Risk: ' + str(pct_current_risk))
    print('Current Portfolio Sharpe: ' + str(current_sharpe) + '\n')

    # calculate expected historical returns and sample covariance matrix
    mu = expected_returns.mean_historical_return(df)
    S = risk_models.sample_cov(df)

    ## optimize for the maximal Sharpe Ratio
    # train EfficientFrontier
    ef = EfficientFrontier(mu, S)

    # calculate and clean (round, etc...) weights
    weights = ef.max_sharpe()
    clean_weights = ef.clean_weights()

    # print portfolio performance
    optimized_return, optimized_risk, optimized_sharpe = ef.portfolio_performance()
    pct_optimized_return = str(round(optimized_return* 100,2) ) + '%'
    pct_optimized_risk = str(round(optimized_risk* 100,2) ) + '%'

    print('Optimized Portfolio Weights: ')
    for symbol,weight in clean_weights.items():
          print(' * ' + symbol,weight)
    print('Optimized Portfolio Return: ' + str(pct_optimized_return))
    print('Optimized Portfolio Risk: ' + str(pct_optimized_risk))
    print('Optimized Portfolio Sharpe: ' + str(round(optimized_sharpe,2)) + '\n')
    
sharpe_optimizer(symbols = ['MMM', 'AAPL', 'T', 'KO', 'JNJ', 'NIO', 'SONO', 'SAS.ST', 'SPYD.DE', 'VUSA.DE'],
                 holdings = np.array([7494, 5728, 3636, 4990, 6887, 2373, 1039, 168, 4582, 6214]))

Current Portfolio Weights: 
 * MMM 0.17383034492356939
 * AAPL 0.13286632182041705
 * T 0.08434042355779267
 * KO 0.11574772099928092
 * JNJ 0.15975041172786528
 * NIO 0.05504395629885644
 * SONO 0.024100577578808194
 * SAS.ST 0.0038969172600960313
 * SPYD.DE 0.10628377908190485
 * VUSA.DE 0.14413954675140916
Current Portfolio Return: 22.39%
Current Portfolio Risk: 16.38%
Current Portfolio Sharpe: 1.24

Optimized Portfolio Weights: 
 * MMM 0.0
 * AAPL 0.30385
 * T 0.0
 * KO 0.0
 * JNJ 0.28179
 * NIO 0.15949
 * SONO 0.0
 * SAS.ST 0.0
 * SPYD.DE 0.25487
 * VUSA.DE 0.0
Optimized Portfolio Return: 38.77%
Optimized Portfolio Risk: 24.93%
Optimized Portfolio Sharpe: 1.47



/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/pypfopt/risk_models.py:68: UserWarning: The covariance matrix is non positive semidefinite. Amending eigenvalues.
  warnings.warn(
